# Face finder — toy example

Sanity-check that InsightFace embeddings can tell the same person apart from a stranger using a few hand-picked photos of LeBron James.

The notebook walks the basic pipeline end-to-end:

1. **Setup** — load the detector (bump detection resolution so small faces in wide shots still detect).
2. **Load + detect** — read four images (two LeBron, two not) and run face detection.
3. **Visualize** — show face crops and bounding boxes to eyeball detection quality.
4. **Compare** — compute cosine similarity between embeddings to see same-person vs different-person scores.

In [ ]:
import sys; sys.path.append("..")  # so we can import utils.py from the project root

import cv2

from utils import app, get_faces, get_embeddings, cosine_similarity, show_faces, show_with_boxes

# bump detection resolution above utils' default 640x640 — these images have
# small/distant faces (basketball crowd shots) that the lower res can miss.
app.prepare(ctx_id=-1, det_size=(1280, 1280))

## Load images and detect faces

Two photos of LeBron and two of unrelated people. We detect faces in each and grab their 512-d embeddings.

In [ ]:
lebron_one = cv2.imread("lebron_one.jpg")
lebron_two = cv2.imread("lebron_two.jpg")
not_lebron_one = cv2.imread("not_lebron_one.jpg")
not_lebron_two = cv2.imread("not_lebron_two.jpg")

In [ ]:
l1_faces = get_faces(lebron_one)
l2_faces = get_faces(lebron_two)
nl1_faces = get_faces(not_lebron_one)
nl2_faces = get_faces(not_lebron_two)

l1_embeddings = get_embeddings(l1_faces)
l2_embeddings = get_embeddings(l2_faces)
nl1_embeddings = get_embeddings(nl1_faces)
nl2_embeddings = get_embeddings(nl2_faces)

print(f"detected faces: lebron_one={len(l1_faces)}, lebron_two={len(l2_faces)}, "
      f"not_lebron_one={len(nl1_faces)}, not_lebron_two={len(nl2_faces)}")

In [ ]:
# inspect detections in lebron_two — useful for spotting low-score / spurious faces
for i, face in enumerate(l2_faces):
    print(f"{i}  bbox={face.bbox}  score={face.det_score:.3f}")

## Visualize detections

`show_faces` plots cropped face tiles; `show_with_boxes` plots the original image with bounding boxes drawn on. Use both to confirm the detector is finding the right faces and nothing spurious.

In [ ]:
show_faces(lebron_one, l1_faces, "lebron_one")
show_faces(not_lebron_two, nl2_faces, "not_lebron_two")

In [ ]:
show_with_boxes(lebron_one, l1_faces, "lebron_one")
show_with_boxes(not_lebron_two, nl2_faces, "not_lebron_two")

## Compare embeddings

Cosine similarity between embeddings is the signal we use to decide same-vs-different person. As a rough calibration: scores **above ~0.5** typically indicate the same person, scores **near 0** indicate different people, and the band in between is the ambiguous zone where you'd want clustering / consensus checks (see `VGGFace2_tests.ipynb`).

In [ ]:
# sanity check: LeBron vs a stranger should score low
diff_person_sim = cosine_similarity(l1_embeddings[0], nl2_embeddings[0])

# LeBron in two different photos should score high
same_person_sim = cosine_similarity(l1_embeddings[0], l2_embeddings[0])

print(f"same person  (lebron_one vs lebron_two):  {same_person_sim:.3f}")
print(f"diff person  (lebron_one vs not_lebron_two): {diff_person_sim:.3f}")